# 05 — Exit: Hold Tetap vs Exit Skor

Notebook 04 menemukan exit skor (skor kemarin < persentil 30%) **kalah dari hold tetap 14 hari** di hampir semua
tahun kecuali 2022. Notebook ini menguji itu dengan benar: stop keras −25% tetap ada, biaya 0,15%/sisi, dan
dilihat juga dari sisi **portofolio** (drawdown), bukan cuma per sinyal.

### Varian (ditetapkan sebelum eksekusi)
| Kode | Exit |
|---|---|
| E0 | **sekarang**: skor < 0,30 ATAU 14 hari ATAU stop −25% |
| E1 | hold tetap **7 hari** + stop −25% |
| E2 | hold tetap **14 hari** + stop −25% |
| E3 | skor < **0,15** (lebih longgar) ATAU 14 hari ATAU stop −25% |

Entry semua sama: top-5 RSI14+RSI7, filter H8b risk-on, masuk open besok.

### Aturan keputusan (ditetapkan sebelum eksekusi)
1. Kandidat = varian dengan **ekspektasi per sinyal tertinggi di dev 2021–2023**.
2. Diadopsi menggantikan E0 hanya kalau di dev **juga**: max drawdown portofolio tidak lebih buruk > 5 poin dari E0
   dan hasil 2022 (bear) tidak lebih buruk > 3 poin dari E0.
3. Lalu dicek di 2024 dan 2025–26. ⚠️ Kedua periode itu **sudah terlihat** di notebook 03–04 (bahkan nb04 sudah
   melihat hold 14 hari di sana), jadi bukan out-of-sample murni. Bukti jujur hanya dari paper trading.

**Portofolio sederhana**: modal dibagi rata ke semua posisi yang sedang terbuka (maks 5 sinyal/hari × lama hold),
tiap posisi baru dapat porsi 1/(5 × hold maks) modal, sisa di USDT; dihitung return harian dan max drawdown.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from data_source.fear_greed import load_fear_greed_index

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
RAW = PROJECT_ROOT / "notebooks" / "data" / "raw"
STABLE = {"BFUSDUSDT", "RLUSDUSDT", "USD1USDT", "USDCUSDT", "USDEUSDT", "USDSUSDT", "UUSDT"}
GOLD, DUP = {"PAXGUSDT", "XAUTUSDT"}, {"WBETHUSDT"}
ALL_SYMS = sorted(p.stem.replace("_1d_full", "") for p in RAW.glob("*_1d_full.csv"))
UNIV = [s for s in ALL_SYMS if s not in STABLE | GOLD | DUP]
BTC, COST = "BTCUSDT", 0.0015
COINS_NB = [c for c in UNIV if c != BTC]

def load(sym):
    df = pd.read_csv(RAW / f"{sym}_1d_full.csv", parse_dates=[0], index_col=0)
    df.columns = [c.lower() for c in df.columns]
    return df

raw = {s: load(s) for s in UNIV}
dates = pd.date_range("2021-01-01", "2026-06-30", freq="D")
close_df = pd.DataFrame({s: raw[s]["close"].reindex(dates) for s in UNIV})
open_df = pd.DataFrame({s: raw[s]["open"].reindex(dates) for s in UNIV})
low_df = pd.DataFrame({s: raw[s]["low"].reindex(dates) for s in UNIV})

fgi = load_fear_greed_index()["fng_value"].reindex(dates).ffill()
btc = close_df[BTC]
market_ok = ((btc > btc.rolling(100, min_periods=100).mean()) | (fgi.rolling(14).mean() > 50)).to_numpy(bool)

def rsi(s, n=14):
    d = s.diff()
    up = d.clip(lower=0).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    return 100 - 100 / (1 + up / dn)

score_df = (close_df.apply(rsi, n=14).rank(axis=1, pct=True) + close_df.apply(rsi, n=7).rank(axis=1, pct=True)) / 2
CLOSE, OPEN, LOW = (df[UNIV].to_numpy(float) for df in (close_df, open_df, low_df))
SCORE = score_df[UNIV].to_numpy(float)
NB_IDX = np.array([UNIV.index(c) for c in COINS_NB])
T, K, HARD_STOP = len(dates), 5, 0.25
print(f"Universe {len(UNIV)} coin, {dates[0].date()} → {dates[-1].date()}")

Universe 40 coin, 2021-01-01 → 2026-06-30


## Mesin evaluasi

Satu fungsi untuk semua varian. Selain return akhir, dicatat **jalur harian** tiap sinyal (return dari harga
sebelumnya ke harga hari itu; hari keluar memakai harga exit) supaya bisa disusun jadi portofolio.

In [2]:
def evaluate(floor, max_hold):
    '''floor=None -> tanpa exit skor (hold tetap). Return: tabel sinyal + matriks return harian portofolio.'''
    rows, port_ret, port_cnt = [], np.zeros(T), np.zeros(T)
    slot = 1.0 / (K * max_hold)
    for i in range(T - 2):
        if not market_ok[i]:
            continue
        row = SCORE[i, NB_IDX]
        valid = ~np.isnan(row)
        if valid.sum() < K:
            continue
        for jj in np.argsort(np.where(valid, row, -np.inf))[::-1][:K]:
            ci = NB_IDX[jj]
            entry = OPEN[i + 1, ci]
            if np.isnan(entry) or entry <= 0:
                continue
            stop = entry * (1 - HARD_STOP)
            exit_px, reason, exit_i = None, None, None
            end = min(i + 1 + max_hold, T)
            for day in range(i + 1, end):
                lo, op = LOW[day, ci], OPEN[day, ci]
                if not np.isnan(lo) and lo <= stop:
                    exit_px = op if (day != i + 1 and not np.isnan(op) and op <= stop) else stop
                    reason, exit_i = "STOP_KERAS", day; break
                if floor is not None and day > i + 1:
                    sc_y = SCORE[day - 1, ci]
                    if not np.isnan(sc_y) and sc_y < floor:
                        exit_px, reason, exit_i = (op if not np.isnan(op) else CLOSE[day, ci]), "SKOR_TURUN", day; break
            if exit_px is None:
                exit_i = end - 1
                exit_px = CLOSE[exit_i, ci]
                if np.isnan(exit_px):
                    continue
                reason = "MAKS_HARI"
            ret = exit_px * (1 - COST) / (entry * (1 + COST)) - 1
            rows.append(dict(i=i, date=dates[i], ret=ret, reason=reason, hold=exit_i - i))
            # jalur harian: hari i+1..exit_i
            prev = entry * (1 + COST)
            for day in range(i + 1, exit_i + 1):
                px = exit_px * (1 - COST) if day == exit_i else CLOSE[day, ci]
                if np.isnan(px):
                    continue
                port_ret[day] += slot * (px / prev - 1)
                prev = px
    sig = pd.DataFrame(rows)
    sig["year"] = sig.date.dt.year
    return sig, pd.Series(port_ret, index=dates)

VARIANTS = {"E0 skor<0,30 (sekarang)": (0.30, 14), "E1 hold 7 hari": (None, 7),
            "E2 hold 14 hari": (None, 14), "E3 skor<0,15": (0.15, 14)}
RES = {name: evaluate(*p) for name, p in VARIANTS.items()}
s0 = RES["E0 skor<0,30 (sekarang)"][0]
chk = s0[(s0.date >= "2024-01-01") & (s0.date < "2026-06-30")]
print(f"Cek reproduksi E0 uji: n={len(chk)} win_rate={(chk.ret > 0).mean():.1%} ekspektasi={chk.ret.mean():+.2%}  (nb02: 2855 / 45.5% / +1.79%)")

Cek reproduksi E0 uji: n=2855 win_rate=45.5% ekspektasi=+1.79%  (nb02: 2855 / 45.5% / +1.79%)


## 1. Per sinyal, per periode

In [3]:
PERIODS = {"dev 2021-23": ("2021-01-01", "2023-12-31"), "2022 (bear)": ("2022-01-01", "2022-12-31"),
           "2024": ("2024-01-01", "2024-12-31"), "2025-26": ("2025-01-01", "2026-06-29")}

def max_dd(r):
    eq = (1 + r).cumprod()
    return (eq / eq.cummax() - 1).min()

rows = []
for name, (sig, pr) in RES.items():
    for pname, (a, b) in PERIODS.items():
        d = sig[(sig.date >= a) & (sig.date <= b)]
        r = pr[a:b]
        rows.append(dict(varian=name, periode=pname, n=len(d), win_rate=(d.ret > 0).mean(), ekspektasi=d.ret.mean(),
                         persentil_5=d.ret.quantile(0.05), rata2_hold=d.hold.mean(),
                         port_return=(1 + r).prod() - 1, port_maxdd=max_dd(r)))
tab = pd.DataFrame(rows).set_index(["varian", "periode"])
show = tab.copy()
for c in ("win_rate",): show[c] = tab[c].map(lambda v: f"{v:.1%}")
for c in ("ekspektasi", "persentil_5", "port_return", "port_maxdd"): show[c] = tab[c].map(lambda v: f"{v:+.2%}")
show["rata2_hold"] = tab.rata2_hold.map(lambda v: f"{v:.1f}")
show

n win_rate ekspektasi persentil_5 rata2_hold port_return port_maxdd
varian                  periode                                                                            
E0 skor<0,30 (sekarang) dev 2021-23  2785    46.4%     +8.87%     -25.22%       11.6   +1066.50%    -58.57%
                        2022 (bear)   130    14.6%    -11.11%     -25.22%       10.5     -19.34%    -21.24%
                        2024         1565    47.6%     +4.61%     -25.22%       11.9     +54.86%    -56.85%
                        2025-26      1290    42.9%     -1.64%     -25.22%       12.3     -25.47%    -37.95%
E1 hold 7 hari          dev 2021-23  2785    46.9%     +4.30%     -25.22%        6.8    +963.19%    -66.10%
                        2022 (bear)   130    23.8%     -7.31%     -25.22%        6.6     -25.06%    -30.63%
                        2024         1565    47.7%     +2.79%     -25.22%        6.8     +89.07%    -64.97%
                        2025-26      1290    42.4%     -0.94%     -25.22%        6.8     -35.69%    -43.49%
E2 hold 14 hari         dev 2021-23  2785    48.2%     +9.14%     -25.22%       12.9   +1165.64%    -62.36%
                        2022 (bear)   130     7.7%    -12.68%     -25.22%       12.2     -21.98%    -24.79%
                        2024         1565    46.5%     +4.33%     -25.22%       12.9     +45.38%    -60.38%
                        2025-26      1290    44.4%     -1.51%     -25.22%       13.1     -27.01%    -37.92%
E3 skor<0,15            dev 2021-23  2785    47.6%     +9.11%     -25.22%       12.4   +1164.29%    -60.54%
                        2022 (bear)   130    12.3%    -11.66%     -25.22%       11.3     -20.27%    -23.14%
                        2024         1565    46.4%     +4.45%     -25.22%       12.6     +50.67%    -59.27%
                        2025-26      1290    43.6%     -1.63%     -25.22%       12.8     -28.07%    -38.24%

## 2. Aturan keputusan

In [4]:
dev = tab.xs("dev 2021-23", level="periode")
bear = tab.xs("2022 (bear)", level="periode")
E0 = "E0 skor<0,30 (sekarang)"
cand = dev.ekspektasi.idxmax()
dd_ok = dev.port_maxdd[cand] >= dev.port_maxdd[E0] - 0.05
bear_ok = bear.ekspektasi[cand] >= bear.ekspektasi[E0] - 0.03
print(f"Kandidat (ekspektasi dev tertinggi): {cand}  ({dev.ekspektasi[cand]:+.2%} vs E0 {dev.ekspektasi[E0]:+.2%})")
print(f"  max DD portofolio dev : {dev.port_maxdd[cand]:+.1%} vs E0 {dev.port_maxdd[E0]:+.1%} -> {'OK' if dd_ok else 'GAGAL'}")
print(f"  ekspektasi 2022       : {bear.ekspektasi[cand]:+.2%} vs E0 {bear.ekspektasi[E0]:+.2%} -> {'OK' if bear_ok else 'GAGAL'}")
ADOPT = cand if (cand != E0 and dd_ok and bear_ok) else E0
print(f"KEPUTUSAN: {ADOPT}")

Kandidat (ekspektasi dev tertinggi): E2 hold 14 hari  (+9.14% vs E0 +8.87%)
  max DD portofolio dev : -62.4% vs E0 -58.6% -> OK
  ekspektasi 2022       : -12.68% vs E0 -11.11% -> OK
KEPUTUSAN: E2 hold 14 hari


## 3. Per tahun (ekspektasi per sinyal) & konsistensi

In [5]:
py = pd.DataFrame({name: sig[sig.date < "2026-06-30"].groupby("year").ret.mean() for name, (sig, _) in RES.items()})
wins = {name: int((py[name] > py[E0]).sum()) for name in py if name != E0}
print("Jumlah tahun (dari 6) varian lebih baik dari E0:", wins)
py.map(lambda v: f"{v:+.2%}")

Jumlah tahun (dari 6) varian lebih baik dari E0: {'E1 hold 7 hari': 3, 'E2 hold 14 hari': 4, 'E3 skor<0,15': 3}


,"E0 skor<0,30 (sekarang)",E1 hold 7 hari,E2 hold 14 hari,"E3 skor<0,15"
year,,,,
2021,+20.26%,+10.04%,+20.61%,+20.57%
2022,-11.11%,-7.31%,-12.68%,-11.66%
2023,+0.99%,+0.46%,+1.36%,+1.25%
2024,+4.61%,+2.79%,+4.33%,+4.45%
2025,-1.62%,-0.99%,-1.50%,-1.60%
2026,-1.75%,-0.71%,-1.56%,-1.79%


## 4. Kurva portofolio per tahun (return & max drawdown)

In [6]:
rows = []
for name, (_, pr) in RES.items():
    for y in range(2021, 2027):
        r = pr[str(y)]
        rows.append(dict(varian=name, tahun=y, ret=(1 + r).prod() - 1, maxdd=max_dd(r)))
py2 = pd.DataFrame(rows).pivot(index="tahun", columns="varian")
py2.map(lambda v: f"{v:+.1%}")

ret                                                               maxdd                                            
varian E0 skor<0,30 (sekarang) E1 hold 7 hari E2 hold 14 hari E3 skor<0,15 E0 skor<0,30 (sekarang) E1 hold 7 hari E2 hold 14 hari E3 skor<0,15
tahun                                                                                                                                         
2021                  +1195.3%       +1218.8%        +1261.4%     +1255.6%                  -26.5%         -35.5%          -27.9%       -27.9%
2022                    -19.3%         -25.1%          -22.0%       -20.3%                  -21.2%         -30.6%          -24.8%       -23.1%
2023                    +11.6%          +7.6%          +19.2%       +17.0%                  -39.9%         -44.7%          -41.6%       -40.8%
2024                    +54.9%         +89.1%          +45.4%       +50.7%                  -56.9%         -65.0%          -60.4%       -59.3%
2025                    -20.6%         -31.2%          -22.6%       -23.2%                  -37.4%         -40.5%          -37.9%       -37.5%
2026                     -6.1%          -6.6%           -5.7%        -6.3%                  -13.3%         -14.9%          -14.0%       -13.7%

## Kesimpulan

Reproduksi E0 cocok (uji 2024–26: 2855 / 45,5% / +1,79%).

**1. Secara aturan, E2 (hold tetap 14 hari + stop −25%) lolos — tapi selisihnya praktis nol.**
| Ekspektasi per sinyal | dev 2021–23 | 2022 | 2024 | 2025–26 |
|---|---|---|---|---|
| E0 skor < 0,30 (sekarang) | +8,87% | −11,11% | **+4,61%** | −1,64% |
| E2 hold 14 hari | **+9,14%** | −12,68% | +4,33% | −1,51% |
| E3 skor < 0,15 | +9,11% | −11,66% | +4,45% | −1,63% |
| E1 hold 7 hari | +4,30% | −7,31% | +2,79% | −0,94% |

E2 unggul +0,27 poin di dev, kalah −0,28 poin di 2024, unggul +0,13 poin di 2025–26, menang di 4 dari 6 tahun.
Itu dalam batas kebetulan. Kesimpulan jujur: **exit skor dan hold 14 hari setara**. Selisih ~1% di notebook 04
ternyata karena return 14 hari di sana dihitung **tanpa stop keras** — bukan karena exit skor buruk (stop −25%
yang menjelaskan selisihnya; tidak diuji lebih jauh supaya jumlah percobaan tidak bertambah). E1 (7 hari) jelas lebih buruk di tahun bagus (2021 +10% vs +20%).

**Keputusan**: E2 **boleh** dipakai karena lebih **sederhana** untuk user (beli, pegang 14 hari, jual; stop −25%)
tanpa perlu memantau skor tiap hari — bukan karena lebih menguntungkan. Ini **tidak memperbaiki** masalah
2025–2026 (tetap −1,5%).

**2. ⚠️ Temuan terpenting: risiko portofolio sangat tinggi di SEMUA varian.** Kalau semua rekomendasi diikuti
(modal dibagi rata ke posisi terbuka):
| Varian | Max drawdown dev | Max DD 2024 | Return 2025 |
|---|---|---|---|
| E0 | −58,6% | −56,9% | −20,6% |
| E2 | −62,4% | −60,4% | −22,6% |

Bahkan di 2024 yang untung +45–55%, modal sempat turun hampir **−60%** di tengah jalan. 5% sinyal terburuk
selalu kena stop keras −25%. Hasil besar dev didominasi 2021 (+1.200%, reli alt sekali seumur).

**3. Arti untuk fitur rekomendasi**: daftar ini layak sebagai **daftar pantauan** (coin alt yang sedang kuat),
**bukan** strategi untuk menaruh seluruh modal. Untuk modal utama, portofolio V2-60 di
`research_signal_trading` (DD uji ±−19%, notebook 19) jauh lebih aman.

**Jumlah percobaan riset rekomendasi**: nb03 (8) + nb04 (3) + nb05 (3) = 14 varian; diadopsi: 1 (E2, setara E0).